### Silver - Entregas

Problemas identificados:
- `destination.state` com 18 variações de grafia para apenas 4 estados
  reais (sigla, nome completo, abreviação, variação de caixa) — mesma
  lógica de normalização de UF aplicada no notebook de regiões
- `carrier.mode` com `Rodoviário`/`rodoviario`/null
- `delivery_status` com `Delivered`/`delivered`/`atrasado`/`cancelled`/
  `in_transit`/null
- timestamps em 2 formatos (`dd/MM/yyyy HH:mm` e ISO)
- `shipped_at = "31/02/2025 10:00"` na entrega `D00021` — data
  estruturalmente inválida (fevereiro não possui 31 dias), resulta em
  null após o parsing, comportamento esperado
- um `order_ref` associado a 2 entregas distintas — não constitui
  duplicidade, e sim reentrega ou entrega parcial; ambas as linhas são
  mantidas
- `cost` com 10 registros contendo o valor `"unknown"` em vez de número.
  Não identificado na análise inicial; a falha de cast ocorria apenas na
  escrita de `fact_pedido_item`, que agrega o custo de frete por pedido.
  Corrigido neste notebook com try_cast.

Utilização de try_to_timestamp em vez de to_timestamp pelo mesmo motivo
descrito no notebook de vendedores (modo ANSI do runtime).

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F

df_entregas_bronze = spark.table(f"{schema_name}.bronze_entregas")
df_map_uf = spark.table(f"{schema_name}.apoio_map_uf")

def normalizar_texto(col):
    c = F.upper(F.trim(col))
    return F.translate(c, "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ", "AAAAAEEEEIIIIOOOOOUUUUC")

In [ ]:
df_entregas_flat = df_entregas_bronze.select(
    F.col("delivery_id"),
    F.upper(F.trim(F.col("order_ref"))).alias("order_id"),
    F.col("carrier.name").alias("carrier_name"),
    F.col("carrier.mode").alias("carrier_mode_raw"),
    F.col("delivery_status").alias("delivery_status_raw"),
    F.col("timestamps.shipped_at").alias("shipped_at_raw"),
    F.col("timestamps.delivered_at").alias("delivered_at_raw"),
    F.col("destination.state").alias("destination_state_raw"),
    F.col("destination.city").alias("destination_city"),
    F.col("cost").cast("string").alias("cost_raw"),
)

df_entregas_flat = df_entregas_flat.withColumn(
    "cost", F.expr("try_cast(cost_raw AS DOUBLE)")
).drop("cost_raw")

In [ ]:
df_entregas_step2 = (
    df_entregas_flat
    .withColumn(
        "carrier_mode",
        F.when(F.lower(F.trim(F.col("carrier_mode_raw"))) == "rodoviario", "Rodoviário")
         .when(F.lower(F.trim(F.col("carrier_mode_raw"))) == "aéreo", "Aéreo")
         .otherwise(F.initcap(F.trim(F.col("carrier_mode_raw"))))
    )
    .withColumn(
        "delivery_status",
        F.when(F.lower(F.trim(F.col("delivery_status_raw"))) == "delivered", "Entregue")
         .when(F.lower(F.trim(F.col("delivery_status_raw"))) == "in_transit", "Em trânsito")
         .when(F.lower(F.trim(F.col("delivery_status_raw"))) == "atrasado", "Atrasado")
         .when(F.lower(F.trim(F.col("delivery_status_raw"))) == "cancelled", "Cancelado")
         .otherwise("Não informado")
    )
)

In [ ]:
df_entregas_step3 = (
    df_entregas_step2
    .withColumn("destination_state_norm", normalizar_texto(F.col("destination_state_raw")))
    .join(df_map_uf, F.col("destination_state_norm") == F.col("variacao_norm"), "left")
)

In [ ]:
df_entregas_step4 = (
    df_entregas_step3
    .withColumn(
        "shipped_at",
        F.coalesce(
            F.expr("try_to_timestamp(shipped_at_raw, 'dd/MM/yyyy HH:mm')"),
            F.expr("try_to_timestamp(shipped_at_raw, \"yyyy-MM-dd'T'HH:mm:ss\")"),
        )
    )
    .withColumn(
        "delivered_at",
        F.coalesce(
            F.expr("try_to_timestamp(delivered_at_raw, 'dd/MM/yyyy HH:mm')"),
            F.expr("try_to_timestamp(delivered_at_raw, \"yyyy-MM-dd'T'HH:mm:ss\")"),
        )
    )
    .withColumn(
        "tempo_entrega_dias",
        F.round((F.col("delivered_at").cast("long") - F.col("shipped_at").cast("long")) / 86400.0, 2)
    )
)

In [ ]:
df_entregas_final = df_entregas_step4.select(
    "delivery_id", "order_id", "carrier_name", "carrier_mode", "delivery_status",
    "shipped_at", "delivered_at", "tempo_entrega_dias",
    F.col("uf_sigla").alias("destino_uf"),
    F.col("regiao_geografica").alias("destino_regiao"),
    "destination_city", "cost",
)

(
    df_entregas_final.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_entregas")
)

print("Bronze:", df_entregas_bronze.count(), "-> Silver:", df_entregas_final.count(), "(nenhuma linha descartada)")

In [ ]:
print("Sem status informado:", df_entregas_final.filter(F.col("delivery_status") == "Não informado").count())
print("Sem tempo de entrega calculado:", df_entregas_final.filter(F.col("tempo_entrega_dias").isNull()).count())
print()
print("Observação: a entrega D00021 possui shipped_at inválido (31/02/2025),")
print("resultando em valor nulo após o parsing — comportamento esperado.")